Ставим зависимости

In [1]:
!pip install datasets evaluate seqeval -q

Загружаем и готовим датасет

In [2]:
import torch
import os
from datasets import load_dataset

raw_datasets = load_dataset("conll2003", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [4]:
from transformers import AutoTokenizer

def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels


def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs


model_checkpoint = "dslim/bert-large-NER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [5]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

ner_feature = raw_datasets["train"].features["ner_tags"]

label_names = ner_feature.feature.names

In [6]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader


data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [7]:
id2label = {
    0: "O",
    1: "B-MISC",
    2: "I-MISC",
    3: "B-PER",
    4: "I-PER",
    5: "B-ORG",
    6: "I-ORG",
    7: "B-LOC",
    8: "I-LOC"
}

def postprocess(predictions, labels):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

Грузим модель

In [8]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    device_map='cuda'
)

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Метод для вычисления размера модели

In [9]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print('Size (MB):', os.path.getsize("temp.p")/1e6)
    os.remove('temp.p')

print_size_of_model(model)

Size (MB): 1330.288643


Ф-ция для нахождения метрик модели

In [10]:
import evaluate
import torch
from tqdm import tqdm

def evaluate_bert_ner(eval_dataloader, model, device):
    metric = evaluate.load("seqeval")
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            batch = {k: v.to(device)for k, v in batch.items()}
            outputs = model(**batch)

        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]

        true_predictions, true_labels = postprocess(predictions, labels)
        metric.add_batch(predictions=true_predictions, references=true_labels)

    results = metric.compute()
    return results['overall_f1']

Ф-ция для нахождения среднего времени работы

In [11]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import time
from tqdm import tqdm


# Среднее время работы
def measure_time(nlp, raw_datasets, index, n):
    tsum = 0
    for i in tqdm(range(n)):
        start = time.perf_counter()
        ner_results = nlp(' '.join(raw_datasets["test"][index]["tokens"]))
        tsum += (time.perf_counter() - start)
    return tsum / n

## Квантизация fp16

In [14]:
import copy

model_copy = copy.deepcopy(model)
quantized_model = model.half() # float16
print(quantized_model.device)

cuda:0


In [15]:
print_size_of_model(quantized_model)

f1_metric = evaluate_bert_ner(eval_dataloader, quantized_model, 'cuda:0')
print(f1_metric)

Size (MB): 665.210883


100%|██████████| 407/407 [00:08<00:00, 45.31it/s]


0.8282051282051281


In [17]:
nlp = pipeline("ner", model=quantized_model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cuda


In [24]:
mean_time_cuda = measure_time(nlp, raw_datasets, 3, 100)
print('Mean time on cuda:', mean_time_cuda*1000, "ms")

Mean time on cuda: 17.081771880001497 ms


CPU

In [32]:
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-large-NER")
quantized_model = AutoModelForTokenClassification.from_pretrained("dslim/bert-large-NER", device_map='cpu')

quantized_model.half()
print(quantized_model.device)

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


cpu


In [34]:
nlp = pipeline("ner", model=quantized_model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cpu


In [38]:
mean_time_cpu = measure_time(nlp, raw_datasets, 3, 100)
print('Mean time on cpu:', mean_time_cpu*1000, "ms")

100%|██████████| 100/100 [02:33<00:00,  1.53s/it]

Mean time on cpu: 1531.421533529981 ms


## Квантизация int8

In [39]:
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-large-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-large-NER", device_map='cpu')

quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [40]:
print_size_of_model(quantized_model)

Size (MB): 424.398425


In [41]:
f1_metric = evaluate_bert_ner(eval_dataloader, quantized_model, 'cpu')
print(f1_metric)

100%|██████████| 407/407 [12:56<00:00,  1.91s/it]


0.7280334728033473


In [42]:
nlp = pipeline("ner", model=quantized_model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cpu


In [45]:
mean_time_cpu = measure_time(nlp, raw_datasets, 3, 100)
print()
print('Mean time on cpu:', mean_time_cpu*1000, "ms")

100%|██████████| 100/100 [00:20<00:00,  4.98it/s]


Mean time on cpu: 199.97147831999428 ms


## Прунинг

cuda

In [46]:
import torch.nn.utils.prune as prune

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-large-NER")
pruned_model = AutoModelForTokenClassification.from_pretrained("dslim/bert-large-NER", device_map='cuda')

parameters_to_prune = tuple(((module, 'weight') for module in pruned_model.modules() if isinstance(module, torch.nn.Linear)))
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,
)

for module, _ in parameters_to_prune:
    prune.remove(module, 'weight')

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [47]:
print_size_of_model(pruned_model)

Size (MB): 1330.288643


In [48]:
f1_metric = evaluate_bert_ner(eval_dataloader, pruned_model, 'cuda')
print(f1_metric)

100%|██████████| 407/407 [00:26<00:00, 15.33it/s]


0.8324783584482207


In [52]:
nlp = pipeline("ner", model=pruned_model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cuda


In [53]:
mean_time_cuda = measure_time(nlp, raw_datasets, 3, 100)
print()
print('Mean time on cuda:', mean_time_cuda*1000, "ms")

100%|██████████| 100/100 [00:01<00:00, 56.54it/s]


Mean time on cuda: 17.58411707000505 ms


cpu

In [12]:
del model

In [18]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import time
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-large-NER")
pruned_model = AutoModelForTokenClassification.from_pretrained("dslim/bert-large-NER")

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
import torch.nn.utils.prune as prune

parameters_to_prune = tuple(((module, 'weight') for module in pruned_model.modules() if isinstance(module, torch.nn.Linear)))
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,
)

for module, _ in parameters_to_prune:
    prune.remove(module, 'weight')

print(pruned_model.device)

In [15]:
pruned_model.cpu()
print(pruned_model.device)

cpu


In [17]:
nlp = pipeline("ner", model=pruned_model, tokenizer=tokenizer, aggregation_strategy="simple", device="cpu")

ValueError: The model has been loaded with `accelerate` and therefore cannot be moved to a specific device. Please discard the `device` argument when creating your pipeline object.

К сожалению, RAM в колабе не хватает, чтобы сделать прунинг на CPU. А сделать на гпу и потом перекинуть на cpu не дает ошибка выше